# 03 · Leakage graph, frozen partitions and training targets

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires audited records and completed duplicate review. Split membership does not change with model seed.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Review graph inputs and document similarity review

In [ ]:
from oncoplate.governance import require_gate
require_gate(cfg,'supervisor_execution')
records=read_table(p['prepared']/"records.csv")
review=p['prepared']/"near_duplicate_review.csv"
edges=read_table(review) if review.exists() else None
print('Records:',len(records),'Proposed near-duplicate pairs:',0 if edges is None else len(edges))
REVIEW_DECISION_REFERENCE = 'Visual review 2026-09-19 by Md Anas Biswas; 8 pairs inspected; 7 rejected as hash collisions; 1 approved: A4F_95200_0046/A4F_98453_0020 duplicate, merging A4F_95200 and A4F_98453'  # Enter the genuine review note/version; not a model-performance decision.
if cfg['mode']!='demo':
    existing=p['prepared']/"duplicate_review_decision.json"
    if REVIEW_DECISION_REFERENCE:write_json(existing,{'reference':REVIEW_DECISION_REFERENCE,'reviewed_at':utcnow()})
    assert existing.exists() and read_json(existing).get('reference'), 'Complete duplicate review and record its reference before freezing.'

## 2. Freeze groups, split and vocabularies

In [ ]:
from oncoplate.pipeline import prepare_study
study,schemas=prepare_study(cfg,edges)
display(study.groupby(['split','domain']).agg(records=('record_id','size'),groups=('group_id','nunique')))
print(json.dumps(read_json(p['prepared']/"split_integrity.json"),indent=2))

## 3. Verify masks, complete-record semantics and unseen labels

In [ ]:
from oncoplate.targets import load_targets
for head in ('independent','joint'):
    t=load_targets(p['prepared']/f'targets_{head}')
    print(head, 'shape:',t['y'].shape,'observed entries:',int(t['mask'].sum()),'labels:',len(t['schema']['labels']))
    display(read_table(p['prepared']/f'targets_{head}'/'unseen_labels.csv').head())

## 4. Generate the reviewed-claim draft from fitting vocabulary
For the new benchmark, complete the review reference before using these rules for assertive outputs.

In [ ]:
from oncoplate.inputs import draft_rules,foundation_states
rule_path=p['private']/"claim_rules.json"
if not rule_path.exists():write_json(rule_path,draft_rules(schemas['joint']))
if cfg['study']['dataset']=='foodnextdb':write_json(p['private']/"inference_states.json",foundation_states(study))
print('Split and target files saved under',p['prepared'])

In [ ]:
MESSAGE = "nb03: duplicate review recorded (1 real cross-participant duplicate), participant splits frozen, 96 groups"

import sys, subprocess
r = subprocess.run([sys.executable, "tools/commit_cell.py", MESSAGE],
                   cwd="/content/drive/MyDrive/OncoPlate_Research/oncoplate-research",
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr, file=sys.stderr); print("exit:", r.returncode)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
